In [3]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 3.2 MB/s eta 0:00:00


In [4]:
import os
from google.colab import userdata
from groq import Groq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

client = Groq(api_key=os.environ["GROQ_API_KEY"])

print("Groq connected successfully!")

Groq connected successfully!


In [5]:
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {
            "role": "user",
            "content": "What is an AI agent? Explain in simple words."
        }
    ]
)

print(response.choices[0].message.content)

**An AI agent is basically a “smart helper” that can see what’s happening around it, decide what to do, and then act to try to reach a goal.**

Think of it like a robot that can look, think, and move:

| What it does | Why it matters |
|--------------|----------------|
| **Perceives** (e.g., reads sensors, looks at a screen, listens to a voice) | It knows the state of the world. |
| **Decides** (runs a tiny brain, often a set of rules or a learned model) | It chooses the next best step. |
| **Acts** (moves a arm, clicks a button, sends a message) | It changes the world or its own internal state. |

If the agent is a chat‑bot, the “perceive” step is reading your message, the “decide” step is running a language model to generate a response, and the “act” step is sending that response back to you.

Key points:

- **Goal‑driven** – every action is aimed at achieving some objective (e.g., answering a question, winning a game, helping a user).
- **Autonomous** – it can operate without human 

In [6]:
def calculator(expression):
    return eval(expression)

print(calculator("25 * 4 + 10"))

110


In [7]:
def ai_agent(question):
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful AI assistant. Answer the user's question clearly."
            },
            {
                "role": "user",
                "content": question
            }
        ]
    )

    return response.choices[0].message.content


print(ai_agent("What is an AI agent? Explain in simple words."))

**An AI agent is a little “mind” that lives inside a computer or robot and does things on its own.**

Think of it like a helper that can:

1. **See or hear** – It receives information from its surroundings (a camera, microphone, sensors, or a database).
2. **Think** – It uses that information to decide what to do next (by following rules, using a trained model, or learning from experience).
3. **Act** – It performs an action that changes the environment (move a robot arm, send a text message, change a game character’s move, etc.).

So, an AI agent is any software or hardware component that can:

- **Observe** its environment,
- **Process** that observation,
- **Take actions** to reach a goal, and
- **Improve** over time if it learns.

**Examples in everyday life**

| What you see | AI agent at work |
|--------------|------------------|
| A smart speaker that answers questions | The voice assistant “listens”, “understands”, then “talks back”. |
| A robot vacuum | It “sees” the floor lay

In [8]:
import json

# Calculator tool
def calculator(expression):
    try:
        return str(eval(expression))
    except Exception as e:
        return f"Error: {e}"


# Tell the AI what tool is available
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate a mathematical expression.",
            "parameters": {
                "type": "object",
                "properties": {
                    "expression": {
                        "type": "string",
                        "description": "The mathematical expression to calculate."
                    }
                },
                "required": ["expression"]
            }
        }
    }
]


# User's question
messages = [
    {
        "role": "user",
        "content": "What is 25 * 4 + 10?"
    }
]


# Ask the AI
response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

assistant_message = response.choices[0].message

# Check whether the AI wants to use the calculator
if assistant_message.tool_calls:

    print("THINK → AI decided to use the calculator")

    messages.append(assistant_message)

    for tool_call in assistant_message.tool_calls:

        print("ACT → Calling calculator")

        arguments = json.loads(tool_call.function.arguments)

        result = calculator(arguments["expression"])

        print("OBSERVE → Calculator result:", result)

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": "calculator",
            "content": result
        })


    # Ask the AI for the final answer
    final_response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages
    )

    print("\nFinal Answer:")
    print(final_response.choices[0].message.content)

else:
    print(assistant_message.content)

110


In [9]:
import json

# Calculator tool
def calculator(expression):
    try:
        result = eval(expression)
        return str(result)
    except:
        return "Invalid calculation"


# Tool definition
tools = [
    {
        "type": "function",
        "function": {
            "name": "calculator",
            "description": "Calculate mathematical expressions",
            "parameters": {
                "type": "object",
                "properties": {
                    "input": {
                        "type": "string",
                        "description": "Mathematical expression to calculate"
                    }
                },
                "required": ["input"]
            }
        }
    }
]


# Agent function
def ai_agent(question):

    messages = [
        {
            "role": "system",
            "content": "You are a helpful AI agent. Use the calculator tool when the user asks for a calculation."
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # Ask the model what to do
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        tools=tools,
        tool_choice="auto"
    )

    message = response.choices[0].message

    # Check whether the agent wants to use a tool
    if message.tool_calls:

        messages.append(message)

        for tool_call in message.tool_calls:

            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            # Execute calculator
            if tool_name == "calculator":
                result = calculator(arguments["input"])

                print("ACT: Calculator")
                print("Calculation:", arguments["input"])
                print("OBSERVE: Result =", result)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": tool_name,
                    "content": result
                })

        # Get final answer
        final_response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            tools=tools,
            tool_choice="none"
        )

        return final_response.choices[0].message.content

    else:
        return message.content


# Test the agent
print(ai_agent("What is 25 * 4 + 10?"))

ACT: Calculator
Calculation: 25 * 4 + 10
OBSERVE: Result = 110
The result is **110**.


In [10]:
import json

# Calculator function
def calculator(expression):
    return str(eval(expression))

# Define the calculator tool
calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Calculate mathematical expressions.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The mathematical expression to calculate."
                }
            },
            "required": ["expression"]
        }
    }
}

# Ask the AI agent
messages = [
    {
        "role": "user",
        "content": "What is 25 * 4 + 10?"
    }
]

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=messages,
    tools=[calculator_tool],
    tool_choice="auto"
)

response_message = response.choices[0].message

# Check whether the agent wants to use the calculator
if response_message.tool_calls:

    messages.append(response_message)

    for tool_call in response_message.tool_calls:

        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name == "calculator":
            result = calculator(arguments["expression"])

        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": function_name,
            "content": result
        })

    # Ask the AI for the final answer
    final_response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        tools=[calculator_tool],
        tool_choice="none"
    )

    print(final_response.choices[0].message.content)

else:
    print(response_message.content)

110


In [11]:
def run_agent(question):

    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        tools=[calculator_tool],
        tool_choice="auto"
    )

    response_message = response.choices[0].message

    # If the agent decides to use a tool
    if response_message.tool_calls:

        messages.append(response_message)

        for tool_call in response_message.tool_calls:

            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if function_name == "calculator":
                result = calculator(arguments["expression"])

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })

        final_response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages
        )

        return final_response.choices[0].message.content

    else:
        return response_message.content


print(run_agent("What is an AI agent?"))

### In a nutshell
An **AI agent** is a software component that **perceives its environment, reasons about what to do, and takes actions that influence that environment** in order to achieve some goal or set of goals.  

The term “agent” stresses that it behaves like an autonomous entity: it can sense, plan, decide, and act—much like a human or animal would—though it does so according to the rules and models built into its code.

---

## Core characteristics

| Property | What it means | Typical examples |
|----------|---------------|-----------------|
| **Perception (Input)** | Receives data from sensors or interfaces (e.g., camera frames, text, sensor streams). | A chatbot receives text; a robot receives lidar scans. |
| **Reasoning / Decision‑Making** | Uses algorithms, models, or learned policies to decide what to do next. | A navigation stack plans a path; a game‑AI chooses a move. |
| **Action (Output)** | Produces a signal that changes the environment or communicates results. | M

In [12]:
def get_weather(city):
    # Simple demo weather tool
    weather_data = {
        "Chennai": "Sunny, 32°C",
        "Coimbatore": "Cloudy, 28°C",
        "Bangalore": "Rainy, 24°C",
        "Mumbai": "Sunny, 30°C"
    }

    return weather_data.get(city, "Weather information not available")


weather_tool = {
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get the current weather information for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "Name of the city"
                }
            },
            "required": ["city"]
        }
    }
}

print("Weather tool created successfully!")

Weather tool created successfully!


In [13]:
import json

# Combine both tools
all_tools = [calculator_tool, weather_tool]


def smart_agent(question):

    messages = [
        {
            "role": "system",
            "content": """
You are an AI agent.
You can use the calculator for mathematical questions.
You can use the weather tool for weather questions.
Choose the appropriate tool when needed.
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    # Ask the AI what action to take
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
        tools=all_tools,
        tool_choice="auto"
    )

    assistant_message = response.choices[0].message

    # If the AI wants to use a tool
    if assistant_message.tool_calls:

        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:

            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            # Calculator
            if function_name == "calculator":
                result = calculator(arguments["expression"])
                print("ACT → Calculator")
                print("OBSERVE →", result)

            # Weather
            elif function_name == "get_weather":
                result = get_weather(arguments["city"])
                print("ACT → Weather tool")
                print("OBSERVE →", result)

            # Send tool result back to AI
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })

        # Generate final response
        final_response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages
        )

        return final_response.choices[0].message.content

    else:
        return assistant_message.content


# Test the agent
print(smart_agent("What is the weather in Coimbatore?"))

ACT → Weather tool
OBSERVE → Cloudy, 28°C
The current weather in Coimbatore is **cloudy** with a temperature of **28 °C**.


In [14]:
print(smart_agent("What is an AI agent? Explain in simple words."))

An **AI agent** is a computer program that can act on its own to achieve a goal. Think of it like a tiny helper robot that sees or receives information, thinks about what to do next, and then performs actions or gives answers.  

Key points in simple terms:

| Feature | What it means |
|---------|----------------|
| **Perception** | It can read data (text, images, sensors). |
| **Goal‑oriented** | It has a target or purpose, like answering a question or navigating a room. |
| **Decision‑making** | It uses rules or learned patterns to decide its next move. |
| **Action** | It can produce text, move a robotic arm, send a message, etc. |
| **Learning (often)** | Many AI agents can improve over time by learning from experience. |

So, an AI agent is essentially a smart, autonomous software “actor” that senses, decides, and acts to accomplish tasks.


In [15]:
print(smart_agent("Calculate 125 * 8 + 50"))

ACT → Calculator
OBSERVE → 1050
1050


In [16]:
print(smart_agent("What is the weather in Coimbatore?"))

ACT → Weather tool
OBSERVE → Cloudy, 28°C
**Coimbatore weather**: Cloudy, 28 °C.


In [17]:
import json

all_tools = [calculator_tool, weather_tool]

def smart_agent(question):

    messages = [
        {
            "role": "system",
            "content": """
You are an AI agent.
You can use the calculator for mathematical questions.
You can use the weather tool for weather questions.
Choose the correct tool when needed.
After getting the tool results, give a clear final answer.
"""
        },
        {
            "role": "user",
            "content": question
        }
    ]

    for step in range(5):

        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            tools=all_tools,
            tool_choice="auto"
        )

        assistant_message = response.choices[0].message

        if not assistant_message.tool_calls:
            return assistant_message.content

        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:

            function_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)

            if function_name == "calculator":

                result = calculator(arguments["expression"])

                print("ACT → Calculator")
                print("OBSERVE →", result)

            elif function_name == "get_weather":

                result = get_weather(arguments["city"])

                print("ACT → Weather tool")
                print("OBSERVE →", result)

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "name": function_name,
                "content": result
            })

    return "The agent could not complete the task."

In [18]:
print(smart_agent("What is 45 * 6, and what is the weather in Coimbatore?"))

ACT → Calculator
OBSERVE → 270
ACT → Weather tool
OBSERVE → Cloudy, 28°C
45 × 6 = 270.  
The current weather in Coimbatore is cloudy with a temperature of 28 °C.


In [19]:
while True:
    question = input("Ask your AI Agent: ")

    if question.lower() == "exit":
        print("AI Agent stopped.")
        break

    answer = smart_agent(question)

    print("\nFINAL ANSWER:")
    print(answer)
    print("-" * 50)

Ask your AI Agent: What is the weather in Coimbatore?
ACT → Weather tool
OBSERVE → Cloudy, 28°C

FINAL ANSWER:
The current weather in Coimbatore is cloudy with a temperature of about 28 °C.
--------------------------------------------------
Ask your AI Agent: exit
AI Agent stopped.


In [20]:
print("===== AI AGENT LOOP =====")

print("THINK → Decide which tool is needed")
print("ACT → Use the selected tool")
print("OBSERVE → Read the tool result")
print("REPEAT → Continue until the task is completed")

print("==========================")

===== AI AGENT LOOP =====
THINK → Decide which tool is needed
ACT → Use the selected tool
OBSERVE → Read the tool result
REPEAT → Continue until the task is completed


# Building My First AI Agent

## AI Agent with Calculator and Weather Tools

This project demonstrates a simple AI agent that can understand user questions,
select appropriate tools, execute actions, observe the results, and provide a
final response.

### Tools Used
- Groq LLM
- Calculator Tool
- Weather Tool
- Google Colab
- Python

## Objective

To build a simple AI agent that can understand user requests, decide whether
a tool is required, use the appropriate tool, observe the result, and provide
a final answer to the user.

## Agent Workflow

The AI agent follows the basic agent loop:

THINK → ACT → OBSERVE → REPEAT

- THINK – Understand the user's request and decide what to do.
- ACT – Select and use the required tool.
- OBSERVE – Receive and process the tool result.
- REPEAT – Continue the process until the task is completed.

## Conclusion

A simple AI agent was successfully built using Python and the Groq LLM.

The agent can understand user questions and select appropriate tools such as
a calculator and a weather tool. It demonstrates the basic AI agent workflow:

THINK → ACT → OBSERVE → REPEAT

The project shows how an LLM can be combined with external tools to perform
tasks instead of only generating text.

In [21]:
print("===== FINAL AI AGENT TEST =====")

print("\nTest 1: Normal Question")
print(smart_agent("What is an AI agent?"))

print("\nTest 2: Calculator")
print(smart_agent("Calculate 125 * 8 + 50"))

print("\nTest 3: Weather")
print(smart_agent("What is the weather in Coimbatore?"))

print("\n===== TEST COMPLETED =====")

===== FINAL AI AGENT TEST =====

Test 1: Normal Question
An **AI agent** is a software system that perceives its environment through inputs (such as sensors, data streams, or user interactions) and acts upon that environment to achieve specific goals or objectives. The key characteristics of an AI agent are:

| Feature | Explanation |
|---------|-------------|
| **Autonomy** | It can make decisions and take actions without constant human intervention. |
| **Reactivity** | It responds to changes in its environment in real‑time or near real‑time. |
| **Proactiveness** | It can plan and take initiatives to fulfill its objectives, not just react. |
| **Learning** | Many agents incorporate learning mechanisms (e.g., machine learning, reinforcement learning) to improve performance over time. |
| **Goal‑oriented** | It has clear objectives or a utility function guiding its actions. |

Examples include:
- **Virtual personal assistants** (Siri, Alexa) that understand speech and perform tasks.
-

## Project Overview

This project demonstrates the development of a simple AI agent using Python,
Groq LLM, and tool calling.

The agent can understand user questions and decide whether to answer directly
or use an available tool. Two tools are implemented:

1. Calculator Tool – performs mathematical calculations.
2. Weather Tool – provides demo weather information for selected cities.

The agent follows the basic workflow:

THINK → ACT → OBSERVE → REPEAT

## Technologies Used

- Python
- Google Colab
- Groq API
- OpenAI GPT-OSS-20B model
- Function/Tool Calling
- Calculator Tool
- Weather Tool

## Result

The AI agent was successfully implemented and tested.

The agent can:
- Answer general questions using the LLM.
- Perform calculations using the Calculator Tool.
- Provide demo weather information using the Weather Tool.
- Select the appropriate tool based on the user's request.
- Follow the THINK → ACT → OBSERVE → REPEAT workflow.

All the final test cases were executed successfully.